## Dataset, DataLoaders, Batches, and Epochs

### Why do we need a Dataset and DataLoader?
Imagine you have 10,000 images for your project. You could load every single image and feed them all into your model at once, but doing so can quickly consume all of your system's memory.

Instead, we usually train neural networks on smaller, manageable groups of data called **batches**.

### Key Terms:
* **Dataset**: Provides access to individual samples and their corresponding labels.
* **DataLoader**: Groups individual samples into batches, handles shuffling, and manages iteration.
* **Batch Size**: The number of training samples processed together in one batch.
* **Epoch**: One complete pass through the entire training dataset.
* **Shuffle**: Randomizes the order of the samples, commonly at the beginning of each training epoch to prevent the model from learning the sequence order.

---

## Building My First Dataset

In [2]:
import torch
from torch.utils.data import TensorDataset

# Four samples, each with two features
X = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0],
    [7.0, 8.0]
])

# one class label per sample
y = torch.tensor([0,1,0,1])

dataset = TensorDataset(X, y)

print("Dataset size:", len(dataset))
print("First sample:", dataset[0])
print("Second sample:", dataset[1])

# In an image dataset it will return (image_tensor, label), while the loss function needs the correct label to learn from it.

Dataset size: 4
First sample: (tensor([1., 2.]), tensor(0))
Second sample: (tensor([3., 4.]), tensor(1))


## Create a DataLoader

Now, let's group our four dataset samples into batches of **size 2** using PyTorch's `DataLoader` class.

In [5]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False
)

for X_batch, y_batch in loader:
  print("Inputs:")
  print(X_batch)

  print("Labels:")
  print(y_batch)

  print("---")

print("="*60)
print("here we can see batches of two clearly..")

Inputs:
tensor([[1., 2.],
        [3., 4.]])
Labels:
tensor([0, 1])
---
Inputs:
tensor([[5., 6.],
        [7., 8.]])
Labels:
tensor([0, 1])
---
here we can see batches of two clearly..


If you set `shuffle=True`, the samples will be presented in a randomized order during each epoch, though their labels will always remain paired with the correct inputs.

### Understanding Batch Size and Epochs

Suppose your dataset contains **100 samples** and you set your **batch size to 20**.

The number of batches in one epoch is calculated as:

$$\text{Batches per Epoch} = \frac{\text{Total Samples}}{\text{Batch Size}} = \frac{100}{20} = 5$$

So, a single epoch contains **5 training batches**.

If you train this model for **10 epochs**, the model will process:

$$5 \text{ batches/epoch} \times 10 \text{ epochs} = 50 \text{ batches in total}$$

*(Assuming every sample is used exactly once per epoch, i.e., `drop_last=False`)*

### Visualizing the Relationship:

```
Dataset
   │
   ├── Epoch 1: Batch 1 → Batch 2 → Batch 3 → Batch 4 → Batch 5
   │
   ├── Epoch 2: Batch 1 → Batch 2 → Batch 3 → Batch 4 → Batch 5
   │
   └── ...
   │
   └── Epoch 10: Batch 1 → Batch 2 → Batch 3 → Batch 4 → Batch 5
```

> **Key Takeaway:** Do not confuse an epoch with a batch. An **epoch** is one complete pass through the entire dataset, whereas a **batch** is a single subset of samples processed together.

---

## Training a Neural Network Using a DataLoader

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X = torch.tensor([
    [1.0, 1.0],
    [1.2, 0.8],
    [-1.0, -1.0],
    [-1.2, -0.8],
    [1.0, -1.0],
    [1.2, -0.8]
])

y = torch.tensor([0, 0, 1, 1, 2, 2])

dataset = TensorDataset(X, y)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(50):
    total_loss = 0.0

    for X_batch, y_batch in loader:
        # Forward pass
        logits = model(X_batch)

        # Calculate loss for this batch
        loss = criterion(logits, y_batch)

        # Backpropagation and update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:2d} | "
            f"Average batch loss: {total_loss / len(loader):.4f}"
        )

Epoch  0 | Average batch loss: 1.1337
Epoch 10 | Average batch loss: 0.7590
Epoch 20 | Average batch loss: 0.4949
Epoch 30 | Average batch loss: 0.2918
Epoch 40 | Average batch loss: 0.1628


This is the standard PyTorch training pattern:

1. **Outer Loop**: Iterates over the specified number of **epochs**.
2. **Inner Loop**: Iterates over the **batches** provided by the `DataLoader`.
3. **Forward Pass**: Each batch is passed through the network to generate predictions (logits).
4. **Loss Calculation**: The loss function compares predictions to the ground truth labels.
5. **Backpropagation**: Computes gradients of the loss with respect to model parameters.
6. **Optimization Step**: The optimizer updates the model's parameters based on these gradients.

As training progresses, the overall loss should generally decrease, though exact run-to-run values may vary due to random initialization and shuffling.

For your CNN (Convolutional Neural Network), the dataset will eventually provide multidimensional image tensors rather than flat, two-dimensional feature vectors.

### Structure of an Image-Classification Batch

A typical batch for image classification might look like this:

* **`X_batch.shape`**: `(32, 3, 224, 224)`

Where the dimensions represent:
* **`32`**: Batch size (number of images processed together)
* **`3`**: Number of color channels (e.g., RGB)
* **`224`**: Height of each image in pixels
* **`224`**: Width of each image in pixels

For a dataset with 10 target classes, the corresponding label batch typically has a flat shape of `(32,)`.

### How the CNN Processes This

The CNN receives the batch tensor, extracts spatial features using convolutional layers, and produces final class logits. The loss function then compares these logits against the 32 true target labels.

We will examine how these image tensors are loaded, resized, normalized, and augmented using transforms when we dive into the computer vision section.